# Hoja de Ejercicios 1

## Imports

In [1]:
from pathlib import Path
import os

from dotenv import load_dotenv
from Bio import SeqIO, Entrez
from Bio.SeqRecord import SeqRecord

## Ejercicio 1: Porcentaje GC

Escribe un programa que tome un fichero FASTA con varias secuencias de ADN y calcule el contenido de GC en porcentaje de cada una y lo muestre por pantalla.

A continuación produzca las cadenas de aminoácidos asociadas a cada secuencia de ADN.

Por último escriba estas secuencias de aminoácidos en un archivo FASTA utilizando como identificadores y descripciones las originales del archivo.

### Rutas y carpeta de datos
Definimos la carpeta `data/ejercicio1` (se crea si no existe) y las rutas del FASTA de entrada y del FASTA de proteínas de salida.

In [2]:
ruta_datos = Path("data/ejercicio1")
ruta_datos.mkdir(parents=True, exist_ok=True)

ruta_entrada = ruta_datos / "secuencias_adn.fasta"
ruta_salida = ruta_datos / "proteinas.fasta"

### Fichero FASTA de entrada
Creamos un archivo FASTA con cuatro secuencias de ADN inventadas, que usaremos como entrada del programa.

In [3]:
contenido = """>gen1 Secuencia inventada uno
ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG
>gen2 Secuencia inventada dos
ATGAAACCCGGGTTTAAACCCGGGTTTTAA
>gen3 Secuencia inventada tres
ATGCGCGCGCGCGCATATATATGCGCGTAA
>gen4 Secuencia inventada cuatro
ATGTTTAAAGGGCCCTTTAAAGGGCCCTGA
"""

with open(ruta_entrada, "w") as f:
    f.write(contenido)

### Porcentaje de GC, traducción y escritura del FASTA
Para cada secuencia del fichero:
1. Calculamos el porcentaje de GC: (G + C) / longitud × 100.
2. La traducimos a aminoácidos con `translate()`. El `*` marca un codón de parada.
3. Guardamos la proteína con el identificador y la descripción originales.

Por último, escribimos todas las proteínas en `proteinas.fasta`.

In [4]:
proteinas = []

for registro in SeqIO.parse(ruta_entrada, "fasta"):
    secuencia = registro.seq.upper()

    porcentaje_gc = (secuencia.count("G") + secuencia.count("C")) / len(secuencia) * 100
    print(f"{registro.id}: {porcentaje_gc:.2f}% GC")

    proteina = secuencia.translate()
    print(f"  Proteína: {proteina}")

    proteinas.append(SeqRecord(proteina, id=registro.id, description=registro.description))

numero = SeqIO.write(proteinas, ruta_salida, "fasta")
print(f"Se han escrito {numero} proteínas en {ruta_salida.as_posix()}")

gen1: 56.41% GC
  Proteína: MAIVMGR*KGAR*
gen2: 43.33% GC
  Proteína: MKPGFKPGF*
gen3: 56.67% GC
  Proteína: MRARAYICA*
gen4: 46.67% GC
  Proteína: MFKGPFKGP*
Se han escrito 4 proteínas en data/ejercicio1/proteinas.fasta


## Ejercicio 2: El sarampión

Escribe un *script* que automatice la descarga del genoma oficial del virus del sarampión.

1. Utiliza Entrez.esearch en la base de datos "nucleotide". Usa la base de datos **secundaria** y el término de búsqueda "Measles morbillivirus[Organism]"

2. Extrae el primer ID (IdList[0]) de los resultados devueltos.

3. Pásale ese ID a Entrez.efetch para descargar el registro en formato FASTA.

4. Imprime por pantalla la descripción del registro y los primeros 100 nucleótidos.

### 1. Búsqueda con `Entrez.esearch`
Buscamos en la base de datos `nucleotide` el término `Measles morbillivirus[Organism]`. Para usar la base de datos **secundaria** (RefSeq, curada) añadimos el filtro `srcdb_refseq[PROP]`. NCBI exige indicar un correo.

In [5]:
load_dotenv()

Entrez.email = os.getenv("ENTREZ_EMAIL")

termino = "Measles morbillivirus[Organism] AND srcdb_refseq[PROP]"
with Entrez.esearch(db="nucleotide", term=termino) as manejador:
    resultado = Entrez.read(manejador)

print(f"Resultados encontrados: {resultado['Count']}")

Resultados encontrados: 1


### 2. Primer ID de los resultados
`Entrez.read` devuelve un diccionario. Cogemos el primer identificador de la lista `IdList`.

In [6]:
id_registro = resultado["IdList"][0]
print(f"ID encontrado: {id_registro}")

ID encontrado: 9626945


### 3. Descarga con `Entrez.efetch`
Le pasamos el ID a `efetch` pidiendo el registro en formato FASTA y lo leemos con `SeqIO.read`, ya que solo hay un registro.

In [7]:
with Entrez.efetch(db="nucleotide", id=id_registro, rettype="fasta", retmode="text") as manejador:
    registro = SeqIO.read(manejador, "fasta")

print(f"Registro descargado: {registro.id} ({len(registro.seq)} nucleótidos)")

Registro descargado: NC_001498.1 (15894 nucleótidos)


### 4. Descripción y primeros 100 nucleótidos
Mostramos la descripción del registro y los 100 primeros nucleótidos de la secuencia.

In [8]:
print(f"Descripción: {registro.description}")
print(f"Primeros 100 nucleótidos: {registro.seq[:100]}")

Descripción: NC_001498.1 Measles virus, complete genome
Primeros 100 nucleótidos: ACCAAACAAAGTTGGGTAAGGATAGATCAATCAATGATCATATTCTAGTACACTTAGGATTCAAGATCCTATTATCAGGGACAAGAGCAGGATTAGGGAT


## Ejercicio 3: La COVID-19

Tienes un archivo GenBank completo, pero solo te interesa saber cuántos genes tiene y cómo se llaman, ignorando el resto de secuencias reguladoras o estructurales.

Utilizando el genoma de referencia del SARS-CoV-2 (código de acceso NC_045512), escribe un programa que:

1. Descarga el archivo en formato GenBank (gb).

2. Recorre la tabla de características (features).

3. Filtra exclusivamente aquellas características cuyo tipo (.type) sea "gene".

4. Extrae el nombre del gen accediendo al diccionario de metadatos (qualifiers.get("gene")).

5. Muestra por pantalla un listado numerado con los nombres de todos los genes encontrados y sus coordenadas de inicio y fin.

### 1. Descarga con `Entrez.efetch`
Pedimos el registro `NC_045512` (genoma de referencia del SARS-CoV-2) a la base de datos `nucleotide` en formato **GenBank** (`gb`) y lo leemos con `SeqIO.read`, ya que solo hay un registro. NCBI exige indicar un correo.

In [9]:
load_dotenv()

Entrez.email = os.getenv("ENTREZ_EMAIL")

with Entrez.efetch(db="nucleotide", id="NC_045512", rettype="gb", retmode="text") as manejador:
    registro = SeqIO.read(manejador, "genbank")

print(f"Registro descargado: {registro.id} ({len(registro.seq)} nucleótidos)")

Registro descargado: NC_045512.2 (29903 nucleótidos)


### 2. Recorrer las características y filtrar los genes
`registro.features` es la tabla de características del registro (genes, CDS, regiones UTR...). Recorremos la tabla y nos quedamos solo con las de tipo `"gene"`.

In [10]:
genes = []

for caracteristica in registro.features:
    if caracteristica.type == "gene":
        genes.append(caracteristica)

print(f"Características de tipo gene: {len(genes)}")

Características de tipo gene: 11


### 3. Nombre y coordenadas de cada gen
El nombre está en el diccionario `qualifiers` (con `.get("gene")`, que devuelve una lista, por eso el `[0]`). Las coordenadas están en `location`. Biopython cuenta desde 0, así que sumamos 1 al inicio para mostrarlas como en GenBank.

In [11]:
for numero, gen in enumerate(genes, start=1):
    nombre = gen.qualifiers.get("gene")[0]
    inicio = gen.location.start + 1
    fin = gen.location.end
    print(f"{numero}. {nombre}: {inicio} - {fin}")

1. ORF1ab: 266 - 21555
2. S: 21563 - 25384
3. ORF3a: 25393 - 26220
4. E: 26245 - 26472
5. M: 26523 - 27191
6. ORF6: 27202 - 27387
7. ORF7a: 27394 - 27759
8. ORF7b: 27756 - 27887
9. ORF8: 27894 - 28259
10. N: 28274 - 29533
11. ORF10: 29558 - 29674


## Ejercicio 4: Bacterias: A la caza de proteínas de resistencia

Las bacterias tienen un “superpoder”: Pueden pasarse pequeños anillos de AND circular llamados plásmidos entre ellas. Estos plásmidos suelen contener genes que codifican proteinas de resistencia a los antibióticos.

Para averiguar contra qué antibióticos se está defendiendo una bacteria mutirresistente el primer paso es **secuenciar** el plásmido. En nuestro caso descargaremos la secuencia de su plásmido, aislaremos los genes codificantes (CDS) y los traduciremos a proteínas.

---

Escribe un script que automatice la caza de proteínas de resistencia.

Utilizaremos el plásmido pT181 del patógeno Staphylococcus aureus, cuyo código de acceso en el NCBI es NC_001371.

El programa debe conectarse al NCBI, descargar el plásmido en formato GenBank (gb), buscar todas las regiones CDS y traducirlas a proteínas.

### 1. Descarga del plásmido en formato GenBank
Descargamos con `Entrez.efetch` el plásmido pT181 (`NC_001371`) en formato `gb` y lo leemos con `SeqIO.read`, ya que solo hay un registro.

In [12]:
with Entrez.efetch(db="nucleotide", id="NC_001371", rettype="gbwithparts", retmode="text") as manejador:
    plasmido = SeqIO.read(manejador, "genbank")

print(f"Plásmido descargado: {plasmido.id} ({len(plasmido.seq)} pb)")
print(f"Primeros 30 nucleótidos: {plasmido.seq[:30]}")

Plásmido descargado: NC_001371.1 (6646 pb)
Primeros 30 nucleótidos: TTCTATGCTCCTATATTGATAAGAATAAAC


### 2. Búsqueda de las regiones CDS
Recorremos la tabla de características (`features`) y nos quedamos solo con las de tipo `CDS`, que son las regiones codificantes.

In [13]:
cds = [caracteristica for caracteristica in plasmido.features if caracteristica.type == "CDS"]

print(f"CDS encontrados: {len(cds)}")

CDS encontrados: 8


### 3. Traducción a proteínas
Para cada CDS extraemos su secuencia de ADN (`extract` tiene en cuenta la hebra y las coordenadas) y la traducimos con la tabla genética bacteriana (11), comprobando que el CDS es completo y eliminando el codón de parada. Mostramos el nombre del producto y su proteína.

In [14]:
for numero, caracteristica in enumerate(cds, start=1):
    producto = caracteristica.qualifiers.get("product", ["desconocido"])[0]
    proteina = caracteristica.extract(plasmido.seq).translate(table=11, cds=True)

    print(f"{numero}. {producto} ({len(proteina)} aa)")
    print(f"   {proteina}")

1. pore-forming bacteriocin colicin E1 (522 aa)
   METAVAYYKDGVPYDDKGQVIITLLNGTPDGSGSGGGGGKGGSKSESSAAIHATAKWSTAQLKKTQAEQAARAKAAAEAQAKAKANRDALTQRLKDIVNEALRHNASRTPSATELAHANNAAMQAEAERLRLAKAEEKARKEAEAAEKAFQEAEQRRKEIEREKAETERQLKLAEAEEKRLAALSEEAKAVEIAQKKLSAAQSEVVKMDGEIKTLNSRLSSSIHARDAEMKTLAGKRNELAQASAKYKELDELVKKLSPRANDPLQNRPFFEATRRRVGAGKIREEKQKQVTASETRINRINADITQIQKAISQVSNNRNAGIARVHEAEENLKKAQNNLLNSQIKDAVDATVSFYQTLTEKYGEKYSKMAQELADKSKGKKIGNVNEALAAFEKYKDVLNKKFSKADRDAIFNALASVKYDDWAKHLDQFAKYLKITGHVSFGYDVVSDILKIKDTGDWKPLFLTLEKKAADAGVSYVVALLFSLLAGTTLGIWGIAIVTGILCSYIDKNKLNTINEVLGI
2. colicin release lysis protein (45 aa)
   MRKRFFVGIFAINLLVGCQANYIPDVQGGTIAPSSSSKLTGIAVQ
3. replication initiation protein (52 aa)
   MHEPPVQPDRCALSGNYRLESNPVRHALTPLAAATGNRISRAMMAQTVLQSS
4. Rop family plasmid primer RNA-binding protein (63 aa)
   MTKQEKTALNMARFIRSQTLTLLEKLNELDADEQADICESLHDHADELYRSCLARFGDDGENL
5. MobC family plasmid mobilization relaxosome protein (107 aa)
   MLTIRVTDDEHARLLERCEGKQLAVWMRRVCLGEPVARSGKLPTLAP

## Ejercicio 5: Algunas proteínas "célebres"

Buscar información, y archivos FASTA asociados, relativa a tres de estas proteínas.

| Hemoglobina | Colágeno | Queratina | Enzima amilasa | Tubulina |
|---|---|---|---|---|
| Mioglobina | Albúmina | Fibrinógeno | Fibrina | Ferritina |
| Zeina | Elastina | Gluteína | Pepsina | Trombina |
| Prolactina | Actina | Miosina | Caseína | Calcitonina |
| Oxitocina | Gammaglobulina | Enzima proteasa | Enzima lipasa | Inmunoglobulina |
| Colágeno | Queratina | Albúmina | Glucagón | Hipocretina |

